In [2]:
import json
import os
from datetime import datetime
from pathlib import Path

import altair as alt
import polars as pl
import requests

from src.utils.date import monthYear

DATA_DIR = Path("../../data/cloudflare/radar")
CLOUDFLARE_API_TOKEN = os.environ.get("CLOUDFLARE_API_TOKEN")

DATA_DIR.mkdir(exist_ok=True, parents=True)

In [3]:
url = "https://api.cloudflare.com/client/v4/radar/http/timeseries_groups/tls_version"
headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {CLOUDFLARE_API_TOKEN}",
}
params = {
    "name": "main",
    "location": "RU",
    "aggInterval": "1d",
}

start_date = datetime.strptime("2024-08-08T00:00:00.000Z", "%Y-%m-%dT%H:%M:%S.%fZ")
end_date = datetime.now()
dates = [
    ("2024-08-08T00:00:00.000Z", "2024-09-07T23:59:59.999Z"),
    ("2024-09-08T00:00:00.000Z", "2024-10-07T23:59:59.999Z"),
    ("2024-10-08T00:00:00.000Z", "2024-11-07T23:59:59.999Z"),
    ("2024-11-08T00:00:00.000Z", "2024-12-04T23:59:59.999Z"),
]

# Store the response in a file (if file exists, don't download it again)
for start, end in dates:
    tls_version_file = DATA_DIR / f"tls_version_{start.split('T')[0]}_{end.split('T')[0]}.json"
    tls_version_metadata_file = DATA_DIR / f"tls_version_metadata_{start.split('T')[0]}_{end.split('T')[0]}.json"

    if not tls_version_file.exists() and not tls_version_metadata_file.exists():
        params["dateStart"] = start
        params["dateEnd"] = end

        response = requests.get(url, headers=headers, params=params)

        # Extract response["result"]["main"]
        tls_version = response.json()["result"]["main"]
        tls_version_metadata = response.json()["result"]["meta"]

        with tls_version_file.open("w") as f:
            json.dump(tls_version, f)
        with tls_version_metadata_file.open("w") as f:
            json.dump(tls_version_metadata, f)
    else:
        print("File already exists, skipping download")

File already exists, skipping download
File already exists, skipping download
File already exists, skipping download
File already exists, skipping download


In [4]:
tls_version_files = list(DATA_DIR.glob("tls_version_*.json"))

df = (
    pl.DataFrame([json.load(f.open()) for f in tls_version_files])
    .select("timestamps", "TLS 1.3", "TLS 1.2", "TLS 1.1", "TLS 1.0", "TLS QUIC")
    .explode(pl.all())
    .with_columns(pl.col("timestamps").cast(pl.Datetime), pl.exclude("timestamps").cast(pl.Float64))
    .rename({"TLS QUIC": "QUIC"})
    .unpivot(index="timestamps", on=["TLS 1.3", "QUIC", "TLS 1.2", "TLS 1.1", "TLS 1.0"])
)

df.head()

timestamps,variable,value
datetime[μs],str,f64
2024-11-08 00:00:00,"""TLS 1.3""",65.698344
2024-11-09 00:00:00,"""TLS 1.3""",64.785425
2024-11-10 00:00:00,"""TLS 1.3""",65.096941
2024-11-11 00:00:00,"""TLS 1.3""",65.483023
2024-11-12 00:00:00,"""TLS 1.3""",65.201702


In [5]:
highlight = (
    alt.Chart(pl.DataFrame({"start": [datetime(2024, 11, 1)], "end": [datetime(2024, 11, 10)]}))
    .mark_rect(opacity=0.3, color="orange")
    .encode(x="start:T", x2="end:T")
)

# Create the plot
chart = (
    alt.Chart(df)
    .properties(
        height=125,
    )
    .mark_line()
    .encode(
        x=alt.X(
            "timestamps:T",
            # title="Date"
            title=None,
            axis=alt.Axis(format=monthYear),
        ),
        y=alt.Y("value:Q", title="Requests (%)"),
        color=alt.Color("variable:N", title="TLS Version"),
    )
)

# Combine the highlight and the chart
final_chart = (highlight + chart).configure_legend(orient="none", direction="horizontal", legendX=-25, legendY=160)

# Save and display the plot
final_chart.save("../../plots/tls_usage_over_time.pdf")
final_chart.show()

alt.LayerChart(...)